In [1]:
import random
import math
import numpy as np
from PIL import Image, ImageDraw

In [2]:
IMAGE_PATH = "girl_pearl_earing.png"

In [3]:
WIDTH = 300
HEIGHT = 400
NUM_TRIANGLES = 100

POP_SIZE = 40
GENERATIONS = 1000
TOURNAMENT_SIZE = 4
MUTATION_RATE = 0.08
ELITE_COUNT = 2

OUTPUT_EVERY = 50

In [4]:
class Triangle:
    def __init__(self):
        self.points = [
            (random.randint(0, WIDTH - 1), random.randint(0, HEIGHT - 1)),
            (random.randint(0, WIDTH - 1), random.randint(0, HEIGHT - 1)),
            (random.randint(0, WIDTH - 1), random.randint(0, HEIGHT - 1)),
        ]

        self.color = (
            random.randint(0, 255),  # R
            random.randint(0, 255),  # G
            random.randint(0, 255),  # B
            random.randint(20, 180), # Alpha
        )

    def copy(self):
        t = Triangle.__new__(Triangle)
        t.points = [p for p in self.points]
        t.color = self.color
        return t

    def mutate(self):
        points = list(self.points)
        color = list(self.color)

        if random.random() < 0.5:
            idx = random.randint(0, 2)
            x, y = points[idx]

            x += random.randint(-30, 30)
            y += random.randint(-30, 30)

            x = max(0, min(WIDTH - 1, x))
            y = max(0, min(HEIGHT - 1, y))

            points[idx] = (x, y)

        else:
            channel = random.randint(0, 3)
            color[channel] += random.randint(-30, 30)

            if channel == 3:
                color[channel] = max(5, min(255, color[channel]))
            else:
                color[channel] = max(0, min(255, color[channel]))

        self.points = points
        self.color = tuple(color)

In [5]:
class Individual:
    def __init__(self):
        self.triangles = [Triangle() for _ in range(NUM_TRIANGLES)]
        self.fitness = None

    def copy(self):
        ind = Individual.__new__(Individual)
        ind.triangles = [t.copy() for t in self.triangles]
        ind.fitness = self.fitness
        return ind

    def render(self):
        image = Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 255))

        for triangle in self.triangles:
            layer = Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))
            draw = ImageDraw.Draw(layer)
            draw.polygon(triangle.points, fill=triangle.color)
            image = Image.alpha_composite(image, layer)

        return image.convert("RGB")

    def evaluate(self, target_array):
        generated = np.asarray(self.render(), dtype=np.float32)
        diff = generated - target_array
        mse = np.mean(diff ** 2)
        rmse = math.sqrt(mse)

        self.fitness = rmse
        return rmse

    def mutate(self):
        for triangle in self.triangles:
            if random.random() < MUTATION_RATE:
                triangle.mutate()

In [6]:
def tournament_selection(population):
    competitors = random.sample(population, TOURNAMENT_SIZE)
    competitors.sort(key=lambda ind: ind.fitness)
    return competitors[0].copy()


def crossover(parent1, parent2):
    child = Individual.__new__(Individual)
    child.triangles = []

    for t1, t2 in zip(parent1.triangles, parent2.triangles):
        if random.random() < 0.5:
            child.triangles.append(t1.copy())
        else:
            child.triangles.append(t2.copy())

    child.fitness = None
    return child

In [7]:
def genetic_algorithm():
    target = Image.open(IMAGE_PATH).convert("RGB")
    target = target.resize((WIDTH, HEIGHT))
    target_array = np.asarray(target, dtype=np.float32)

    population = [Individual() for _ in range(POP_SIZE)]

    for ind in population:
        ind.evaluate(target_array)

    best = min(population, key=lambda ind: ind.fitness).copy()

    for generation in range(GENERATIONS):
        population.sort(key=lambda ind: ind.fitness)

        if population[0].fitness < best.fitness:
            best = population[0].copy()

        print(f"Generation {generation}: Best RMSE = {best.fitness:.4f}")

        if generation % OUTPUT_EVERY == 0:
            best.render().save(f"output_generation_{generation}.png")

        new_population = []

        for i in range(ELITE_COUNT):
            new_population.append(population[i].copy())

        while len(new_population) < POP_SIZE:
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)

            child = crossover(parent1, parent2)
            child.mutate()
            child.evaluate(target_array)

            new_population.append(child)

        population = new_population

    best.render().save("final_100_triangles.png")
    print("Saved final image as final_100_triangles.png")

In [8]:
genetic_algorithm()

Generation 0: Best RMSE = 65.8898
Generation 1: Best RMSE = 65.4016
Generation 2: Best RMSE = 63.5655
Generation 3: Best RMSE = 61.5817
Generation 4: Best RMSE = 59.9548
Generation 5: Best RMSE = 59.6599
Generation 6: Best RMSE = 58.6650
Generation 7: Best RMSE = 57.5270
Generation 8: Best RMSE = 57.4802
Generation 9: Best RMSE = 55.9696
Generation 10: Best RMSE = 55.9696
Generation 11: Best RMSE = 55.5395
Generation 12: Best RMSE = 54.5623
Generation 13: Best RMSE = 54.3009
Generation 14: Best RMSE = 54.2089
Generation 15: Best RMSE = 53.2154
Generation 16: Best RMSE = 52.9038
Generation 17: Best RMSE = 52.4680
Generation 18: Best RMSE = 52.2016
Generation 19: Best RMSE = 52.0704
Generation 20: Best RMSE = 51.8332
Generation 21: Best RMSE = 51.5915
Generation 22: Best RMSE = 50.8766
Generation 23: Best RMSE = 50.7458
Generation 24: Best RMSE = 50.5798
Generation 25: Best RMSE = 50.0510
Generation 26: Best RMSE = 49.8875
Generation 27: Best RMSE = 49.7201
Generation 28: Best RMSE = 49.